# Train the product detector (Objective 1)

Shopping robot - computer vision. Run the cells from top to bottom.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.

What this does: gets the code and your labelled photos, checks the dataset, trains YOLOv8n,
scores it on the test photos, converts it for the Raspberry Pi, and saves everything to your Google Drive.

## 1. Connect Google Drive (results are saved there so they are not lost)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_TO = '/content/drive/MyDrive/FYP_vision'   # change if you like
import os; os.makedirs(SAVE_TO, exist_ok=True)

## 2. Install YOLO and get the project code

In [ ]:
!pip install -q ultralytics roboflow
!rm -rf /content/FYPROJECT && git clone -q https://github.com/joan-ouma/FYPROJECT.git /content/FYPROJECT
%cd /content/FYPROJECT/vision

## 3. Get the dataset — use ONE of the two options

**Option A – straight from Roboflow:** in Roboflow open your project → Versions → Export → format **YOLOv8** →
"show download code". Copy your API key, workspace, project and version number into the cell below.

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="PASTE_YOUR_API_KEY")
project = rf.workspace("PASTE_WORKSPACE").project("PASTE_PROJECT")
dataset = project.version(1).download("yolov8", location="/content/dataset")

**Option B – from a ZIP on your Google Drive** (exported from Roboflow as YOLOv8):

In [ ]:
# !rm -rf /content/dataset && mkdir -p /content/dataset
# !unzip -q "/content/drive/MyDrive/FYP_vision/dataset.zip" -d /content/dataset

## 4. Check the dataset (names, order, counts) — fix any PROBLEM before training

In [ ]:
!python training/check_dataset.py /content/dataset --write-yaml

## 5. Train (about 20–40 minutes on a T4 for ~150 photos and 100 epochs)

In [ ]:
RUN = 'run1'
NOTES = 'first run'   # write what is new in this run, e.g. 'added 40 egg photos at night'
!python training/train.py --data /content/dataset/data_checked.yaml --epochs 100 --name {RUN} --notes "{NOTES}"

## 6. Look at the confusion matrix (which products get mixed up)

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob(f'/content/FYPROJECT/vision/runs/{RUN}_*/confusion_matrix*.png')):
    display(Image(filename=f, width=700))

## 7. Convert for the Raspberry Pi (NCNN) and check it still scores the same

In [ ]:
!python training/export_model.py --format ncnn --imgsz 640
!python training/evaluate.py --weights models/best_ncnn_model --data /content/dataset/data_checked.yaml

## 8. Save everything to Google Drive

In [ ]:
!cp -r models training/results training/training_log.csv "{SAVE_TO}/" 2>/dev/null; ls -R "{SAVE_TO}" | head -30
print('Copy models/best_ncnn_model to the Pi (vision/models/) and put training/results in GitHub.')